# Day 83 — Identity & least privilege for agents

Give each agent identity the **smallest** set of scopes it needs, deny everything else, and
require human approval for high-blast actions — all audited. This is the capstone's core
control. Uses [`shared.policy`](../../shared/policy.py). > ✅ Offline.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
from shared.policy import PolicyGate, PolicyDenied

gate = PolicyGate(
    tool_scopes={"read_logs": "logs:read", "deploy": "prod:deploy"},
    granted={"logs:read"},          # this agent may read logs, NOT deploy
    high_blast={"deploy"},
    approver=lambda tool, args: False,   # demo: human always refuses
)

def run(tool, arg=None):
    # TODO: gate.authorize(tool, arg) in try/except PolicyDenied -> return f"DENIED: {exc}";
    #       on success return f"ran {tool}"
    raise NotImplementedError

# print(run("read_logs")); print(run("deploy")); print(run("rm_rf")); gate.audit.dump()

## 🔒 Solution

In [ ]:
from shared.policy import PolicyGate, PolicyDenied

gate = PolicyGate(
    tool_scopes={"read_logs": "logs:read", "deploy": "prod:deploy"},
    granted={"logs:read"},
    high_blast={"deploy"},
    approver=lambda tool, args: False,
)

def run(tool, arg=None):
    try:
        gate.authorize(tool, arg)
    except PolicyDenied as exc:
        return f"DENIED: {exc}"
    return f"ran {tool}"

print(run("read_logs"))   # allowed
print(run("deploy"))      # high-blast -> human refused
print(run("rm_rf"))       # unknown -> deny by default
print("\n--- audit ---")
gate.audit.dump()